# Pooling Generation (Qrels)
This notebook generates the evaluation pool by extracting the top results from BM25, Dense, and Hybrid search for a set of test queries.

In [ ]:
import pandas as pd
from search_engine import BM25Searcher, DenseSearcher, HybridSearcher

# 50 TEST QUERIES CATEGORIZED 
TEST_QUERIES = [
    # Tramas, conceptos y emociones sin nombres propios
    "A story of survival in extreme cold weather and snow",
    "Psychological tension and family secrets between relatives",
    "High school students forming a music band and dealing with youth problems",
    "A journey through outer space to find a new habitable planet for humanity",
    "Documentary showing the history of jazz music and its cultural impact",
    "A detective dealing with his own inner demons while chasing a serial killer",
    "Romantic relationship between people from completely different social classes",
    "A struggle against an oppressive dystopian government in a near future",
    "Feel-good comedy about an unexpected friendship between a senior and a teenager",
    "Athletes overcoming severe physical injuries to win a championship",
    "A house haunted by malevolent spirits that terrorize a family",
    "Time travel causing complex paradoxes and altering historical events",
    "An deep dive into the lives of wild animals in the African savannah",
    "A lawyer fighting against a corrupt corporation to protect poor citizens",
    "An artist struggling with writer's block and losing their mind",
    "A classic revenge story where a protagonist hunts down former betrayers",
    "Coming of age story of a young boy finding his identity in a small town",

    # Directores, actores, combinaciones
    "Movies directed by Steven Spielberg",
    "Movies directed by Martin Scorsese",
    "Films starring Samuel L. Jackson",
    "Movies starring Danny Trejo",
    "Films starring Shah Rukh Khan",
    "Movies starring Amitabh Bachchan",
    "Films starring Nicolas Cage",
    "Movies starring Bruce Willis",
    "Movies directed by Robert Vince",
    "Films directed by Paul Hoen",
    "Movies starring Anupam Kher",
    "Films starring Maggie Binkley",
    "Movies starring Paresh Rawal",
    "Films starring Naseeruddin Shah",
    "Movies starring Akshay Kumar",
    "Films directed by John Lasseter",
    "Movies starring Om Puri",

    # Entidad + Concepto
    "Action movies starring Samuel L. Jackson",
    "Science fiction movies directed by Steven Spielberg",
    "Animated films directed by John Lasseter",
    "Documentaries featuring Elon Musk",
    "Drama movies starring Nicolas Cage",
    "Romantic comedy dramas starring Shah Rukh Khan",
    "Horror movies starring Danny Trejo",
    "Crime drama thriller directed by Martin Scorsese",
    "Comedy movies starring Bruce Willis",
    "Action thrillers starring Amitabh Bachchan",
    "Sports documentary starring or about athletes",
    "Family comedy directed by Robert Vince",
    "Musical comedy directed by Paul Hoen",
    "Comedy movies starring Anupam Kher",
    "Drama movies starring Naseeruddin Shah",
    "Action movies starring Akshay Kumar"
]

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   --- ------------------------------------ 1.3/16.1 MB 10.0 MB/s eta 0:00:02
   --------- ------------------------------ 3.7/16.1 MB 10.2 MB/s eta 0:00:02
   ------------- -------------------------- 5.5/16.1 MB 9.4 MB/s eta 0:00:02
   ------------------- -------------------- 7.9/16.1 MB 9.9 MB/s eta 0:00:01
   ------------------------ --------------- 10.0/16.1 MB 9.9 MB/s eta 0:00:01
   ----------------------------- ---------- 12.1/16.1 MB 10.0 MB/s eta 0:00:01
   ----------------------------------- ---- 14.4/16.1 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------  16.0/16.1 MB 10.2 MB/s eta 0:00:01
   ---------------------------------------- 16.1/16.1 MB 9.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Load the dataframe with embeddings
print("Loading dataframe...")
movies = pd.read_pickle('movies_with_embeddings.pkl')

# Initialize searchers
bm25_searcher = BM25Searcher(movies)
dense_searcher = DenseSearcher(movies)
hybrid_searcher = HybridSearcher(bm25_searcher, dense_searcher)


Loading dataframe...
Loading precalculated BM25 index from bm25_index.pkl...
BM25 Index loaded successfully.
Loading SentenceTransformer model (paraphrase-multilingual-MiniLM-L12-v2)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading precalculated FAISS index from faiss_index.bin...
FAISS Index and ID map loaded successfully.


In [ ]:
# Generate Pool
pool_rows = []
top_k = 10
global_id = 0

for query in TEST_QUERIES:
    print(f"Processing query: '{query}'")
    bm25_res = bm25_searcher.search(query, top_k=top_k)
    dense_res = dense_searcher.search(query, top_k=top_k)
    hybrid_res = hybrid_searcher.search(query, top_k=top_k)
    
    unique_titles = set()
    for res in bm25_res + dense_res + hybrid_res:
        unique_titles.add(res['title'])
        
    for title in sorted(unique_titles):
        pool_rows.append({
            'id': global_id,
            'query': query,
            'movie_title': title,
            'relevance': ''
        })
        global_id += 1

pool_df = pd.DataFrame(pool_rows)
print(f"\nGenerated pool with {len(pool_df)} unique query-movie pairs.")

# Formatter for copy-pasting to the LLM 
movie_desc_dict = dict(zip(movies['title'], movies['content_narrative']))

prompt_content = []
prompt_header = """Instructions for the AI Labeler:
You are an expert movie relevance annotator.
For each item below, you are given a unique ID, a Search Query, a Movie Title, and a Narrative Content (synopsis and details).
Grade the relevance of the movie to the query on a scale of 0 to 5:
0: Completely irrelevant (no connection whatsoever)
1: Barely relevant (very weak or accidental keyword overlap)
2: Somewhat relevant (shares some elements like genre or minor themes, but not matching core intent)
3: Relevant (matches the main concepts/entities of the search query, but might miss specific aspects)
4: Highly relevant (excellent match to the query, covers the search intent very well)
5: Perfect match (exact match to the query, or is the precise movie/franchise/subject requested)

Return your response as a simple list where each line has the format:
ID: Score
Example:
0: 5
1: 0
2: 3

Here are the items to grade:
"""
prompt_content.append(prompt_header)

for _, row in pool_df.iterrows():
    row_id = row['id']
    q = row['query']
    title = row['movie_title']
    desc = movie_desc_dict.get(title, "No description available.")
    
    item_str = f"ID: {row_id}\nQuery: {q}\nMovie Title: {title}\nNarrative Content: {desc}\n--------------------------------------------------\n"
    prompt_content.append(item_str)

output_prompt_path = 'prompt_for_ai.txt'
with open(output_prompt_path, 'w', encoding='utf-8') as f:
    f.writelines(prompt_content)

print(f"AI Prompt successfully exported to '{output_prompt_path}'.")
print("Preview of the first 3 items:")
print("".join(prompt_content[1:4]))

Processing query: 'A story of survival in extreme cold weather and snow'
Processing query: 'Psychological tension and family secrets between relatives'
Processing query: 'High school students forming a music band and dealing with youth problems'
Processing query: 'A journey through outer space to find a new habitable planet for humanity'
Processing query: 'Documentary showing the history of jazz music and its cultural impact'
Processing query: 'A detective dealing with his own inner demons while chasing a serial killer'
Processing query: 'Romantic relationship between people from completely different social classes'
Processing query: 'A struggle against an oppressive dystopian government in a near future'
Processing query: 'Feel-good comedy about an unexpected friendship between a senior and a teenager'
Processing query: 'Athletes overcoming severe physical injuries to win a championship'
Processing query: 'A house haunted by malevolent spirits that terrorize a family'
Processing query

In [ ]:
# Results Importer
import re
import os

ai_output_raw = """
1: 1
2: 5
3: 0
4: 0
5: 2
6: 0
7: 4
8: 4
9: 2
10: 2
11: 0
12: 3
13: 1
14: 1
15: 2
16: 1
17: 2
18: 4
19: 1
20: 5
21: 1
22: 3
23: 4
24: 4
25: 1
26: 3
27: 2
28: 2
29: 3
30: 0
31: 3
32: 1
33: 4
34: 4
35: 5
36: 3
37: 2
38: 2
39: 4
40: 1
41: 1
42: 1
43: 5
44: 0
45: 1
46: 0
47: 4
48: 2
49: 2
50: 5
51: 2
52: 1
53: 2
54: 1
55: 4
56: 0
57: 2
58: 1
59: 3
60: 1
61: 1
62: 1
63: 1
64: 1
65: 3
66: 4
67: 0
68: 2
69: 2
70: 1
71: 1
72: 0
73: 0
74: 4
75: 1
76: 1
77: 2
78: 3
79: 0
80: 3
81: 2
82: 5
83: 5
84: 2
85: 0
86: 2
87: 0
88: 3
89: 5
90: 1
91: 0
92: 1
93: 0
94: 1
95: 2
96: 5
97: 0
98: 0
99: 3
100: 3
101: 0
102: 1
103: 1
104: 1
105: 4
106: 1
107: 1
108: 3
109: 1
110: 2
111: 3
112: 4
113: 1
114: 3
115: 3
116: 2
117: 3
118: 3
119: 3
120: 3
121: 3
122: 2
123: 4
124: 3
125: 5
126: 1
127: 5
128: 0
129: 3
130: 5
131: 0
132: 1
133: 1
134: 1
135: 2
136: 1
137: 1
138: 3
139: 5
140: 4
141: 5
142: 5
143: 0
144: 5
145: 0
146: 4
147: 0
148: 1
149: 2
150: 3
151: 2
152: 1
153: 4
154: 3
155: 3
156: 4
157: 0
158: 1
159: 4
160: 2
161: 1
162: 0
163: 0
164: 1
165: 3
166: 2
167: 4
168: 2
169: 1
170: 5
171: 2
172: 1
173: 2
174: 2
175: 0
176: 3
177: 0
178: 5
179: 0
180: 0
181: 0
182: 0
183: 0
184: 0
185: 1
186: 2
187: 0
188: 0
189: 1
190: 1
191: 1
192: 0
193: 1
194: 2
195: 0
196: 0
197: 3
198: 4
199: 1
200: 2
201: 1
202: 0
203: 1
204: 3
205: 0
206: 5
207: 1
208: 1
209: 0
210: 1
211: 2
212: 2
213: 5
214: 2
215: 1
216: 1
217: 4
218: 5
219: 5
220: 2
221: 3
222: 3
223: 3
224: 3
225: 2
226: 1
227: 5
228: 3
229: 3
230: 3
231: 2
232: 3
233: 5
234: 5
235: 4
236: 4
237: 5
238: 3
239: 0
240: 2
241: 0
242: 2
243: 4
244: 0
245: 0
246: 4
247: 5
248: 2
249: 5
250: 2
251: 3
252: 3
253: 2
254: 4
255: 4
256: 0
257: 1
258: 1
259: 2
260: 0
261: 5
262: 5
263: 1
264: 0
265: 1
266: 1
267: 0
268: 1
269: 5
270: 0
271: 1
272: 5
273: 4
274: 0
275: 1
276: 4
277: 2
278: 3
279: 3
280: 1
281: 0
282: 1
283: 1
284: 1
285: 2
286: 4
287: 1
288: 1
289: 1
290: 0
291: 2
292: 4
293: 1
294: 3
295: 3
296: 2
297: 2
298: 4
299: 2
300: 5
301: 0
302: 1
303: 3
304: 2
305: 0
306: 2
307: 4
308: 3
309: 2
310: 0
311: 2
312: 2
313: 0
314: 1
315: 1
316: 2
317: 2
318: 1
319: 5
320: 3
321: 5
322: 2
323: 3
324: 4
325: 3
326: 3
327: 3
328: 0
329: 2
330: 3
331: 3
332: 3
333: 3
334: 3
335: 2
336: 0
337: 3
338: 1
339: 5
340: 3
341: 2
342: 2
343: 1
344: 4
345: 4
346: 3
347: 3
348: 2
349: 4
350: 3
351: 1
352: 4
353: 0
354: 1
355: 0
356: 2
357: 4
358: 2
359: 2
360: 5
361: 0
362: 1
363: 5
364: 5
365: 5
366: 5
367: 5
368: 1
369: 0
370: 0
371: 0
372: 5
373: 0
374: 5
375: 5
376: 0
377: 0
378: 0
379: 5
380: 0
381: 5
382: 0
383: 5
384: 5
385: 5
386: 0
387: 5
388: 0
389: 5
390: 5
391: 0
392: 0
393: 5
394: 5
395: 5
396: 5
397: 5
398: 5
399: 1
400: 1
401: 0
402: 1
403: 1
404: 0
405: 0
406: 0
407: 0
408: 1
409: 5
410: 1
411: 5
412: 1
413: 0
414: 5
415: 1
416: 1
417: 1
418: 1
419: 1
420: 1
421: 5
422: 5
423: 0
424: 0
425: 1
426: 1
427: 1
428: 1
429: 1
430: 0
431: 1
432: 0
433: 0
434: 1
435: 0
436: 0
437: 1
438: 1
439: 0
440: 0
441: 0
442: 0
443: 0
444: 0
445: 2
446: 5
447: 0
448: 5
449: 0
450: 0
451: 5
452: 5
453: 0
454: 0
455: 0
456: 5
457: 0
458: 0
459: 0
460: 0
461: 0
462: 5
463: 0
464: 5
465: 0
466: 5
467: 5
468: 0
469: 1
470: 1
471: 1
472: 0
473: 5
474: 0
475: 0
476: 0
477: 0
478: 0
479: 5
480: 1
481: 5
482: 5
483: 0
484: 1
485: 0
486: 0
487: 5
488: 5
489: 0
490: 0
491: 5
492: 0
493: 0
494: 0
495: 0
496: 0
497: 0
498: 5
499: 5
500: 5
501: 5
502: 5
503: 0
504: 0
505: 0
506: 0
507: 5
508: 5
509: 5
510: 0
511: 5
512: 5
513: 5
514: 0
515: 5
516: 0
517: 0
518: 0
519: 0
520: 1
521: 0
522: 0
523: 0
524: 5
525: 5
526: 5
527: 5
528: 5
529: 5
530: 5
531: 0
532: 5
533: 0
534: 0
535: 5
536: 5
537: 1
538: 0
539: 5
540: 5
541: 0
542: 5
543: 0
544: 0
545: 5
546: 5
547: 5
548: 5
549: 0
550: 5
551: 0
552: 0
553: 0
554: 5
555: 5
556: 0
557: 5
558: 5
559: 0
560: 0
561: 0
562: 0
563: 0
564: 0
565: 0
566: 0
567: 5
568: 0
569: 5
570: 0
571: 0
572: 0
573: 0
574: 0
575: 0
576: 0
577: 0
578: 0
579: 0
580: 0
581: 0
582: 0
583: 0
584: 1
585: 0
586: 0
587: 0
588: 1
589: 1
590: 1
591: 1
592: 0
593: 0
594: 0
595: 1
596: 0
597: 1
598: 0
599: 0
600: 0
601: 0
602: 0
603: 0
604: 0
605: 0
606: 0
607: 1
608: 0
609: 0
610: 0
611: 0
612: 0
613: 0
614: 0
615: 0
616: 0
617: 0
618: 0
619: 0
620: 0
621: 0
622: 0
623: 0
624: 0
625: 0
626: 0
627: 0
628: 0
629: 0
630: 0
631: 0
632: 0
633: 0
634: 0
635: 0
636: 0
637: 1
638: 5
639: 0
640: 0
641: 5
642: 5
643: 5
644: 0
645: 0
646: 0
647: 5
648: 5
649: 0
650: 0
651: 0
652: 0
653: 1
654: 0
655: 0
656: 0
657: 0
658: 1
659: 1
660: 0
661: 5
662: 5
663: 5
664: 0
665: 5
666: 5
667: 5
668: 5
669: 5
670: 1
671: 5
672: 0
673: 5
674: 0
675: 0
676: 0
677: 5
678: 0
679: 0
680: 0
681: 0
682: 5
683: 1
684: 1
685: 1
686: 1
687: 1
688: 1
689: 1
690: 1
691: 0
692: 0
693: 0
694: 0
695: 1
696: 0
697: 0
698: 5
699: 0
700: 0
701: 0
702: 0
703: 5
704: 0
705: 0
706: 1
707: 0
708: 0
709: 0
710: 0
711: 0
712: 0
713: 0
714: 5
715: 0
716: 0
717: 0
718: 3
719: 0
720: 0
721: 5
722: 0
723: 0
724: 0
725: 0
726: 3
727: 2
728: 2
729: 2
730: 0
731: 3
732: 3
733: 3
734: 5
735: 2
736: 2
737: 2
738: 0
739: 2
740: 2
741: 3
742: 0
743: 3
744: 3
745: 3
746: 5
747: 5
748: 5
749: 5
750: 5
751: 5
752: 5
753: 0
754: 0
755: 5
756: 0
757: 5
758: 2
759: 2
760: 5
761: 0
762: 1
763: 1
764: 5
765: 5
766: 5
767: 2
768: 1
769: 1
770: 0
771: 5
772: 1
773: 0
774: 1
775: 1
776: 1
777: 1
778: 1
779: 1
780: 5
781: 1
782: 1
783: 1
784: 1
785: 1
786: 1
787: 1
788: 3
789: 5
790: 1
791: 1
792: 1
793: 0
794: 1
795: 3
796: 0
797: 3
798: 1
799: 0
800: 3
801: 1
802: 1
803: 1
804: 5
805: 5
806: 2
807: 3
808: 2
809: 3
810: 1
811: 2
812: 4
813: 2
814: 4
815: 2
816: 1
817: 2
818: 2
819: 2
820: 2
821: 1
822: 1
823: 2
824: 2
825: 2
826: 1
827: 1
828: 1
829: 3
830: 3
831: 1
832: 1
833: 1
834: 0
835: 1
836: 0
837: 1
838: 1
839: 1
840: 1
841: 1
842: 1
843: 1
844: 1
845: 1
846: 1
847: 1
848: 1
849: 3
850: 2
851: 2
852: 2
853: 5
854: 5
855: 2
856: 1
857: 1
858: 5
859: 2
860: 2
861: 2
862: 5
863: 2
864: 5
865: 5
866: 1
867: 2
868: 1
869: 3
870: 3
871: 3
872: 3
873: 1
874: 1
875: 1
876: 0
877: 1
878: 3
879: 5
880: 3
881: 1
882: 3
883: 3
884: 3
885: 1
886: 1
887: 3
888: 1
889: 3
890: 0
891: 1
892: 5
893: 1
894: 5
895: 0
896: 3
897: 2
898: 2
899: 2
900: 0
901: 0
902: 2
903: 0
904: 0
905: 0
906: 2
907: 4
908: 4
909: 5
910: 4
911: 5
912: 4
913: 4
914: 5
915: 5
916: 5
917: 5
918: 5
919: 5
920: 4
921: 4
922: 2
923: 2
924: 5
925: 5
926: 5
927: 1
928: 3
929: 2
930: 0
931: 1
932: 5
933: 1
934: 1
935: 5
936: 5
937: 5
938: 5
939: 5
940: 5
941: 5
942: 2
943: 2
944: 2
945: 4
946: 5
947: 1
948: 1
949: 1
950: 2
951: 5
952: 3
953: 1
954: 1
955: 5
956: 3
957: 1
958: 1
959: 3
960: 3
961: 1
962: 1
963: 1
964: 1
965: 4
966: 3
967: 1
968: 3
969: 1
970: 5
971: 5
972: 1
973: 0
974: 1
975: 0
976: 1
977: 1
978: 1
979: 1
980: 1
981: 3
982: 2
983: 1
984: 0
985: 1
986: 5
987: 1
988: 1
989: 1
990: 1
991: 1
992: 1
993: 1
994: 1
995: 1
996: 1
997: 1
998: 1
999: 5
1000: 1
1001: 1
1002: 1
1003: 0
1004: 1
1005: 0
1006: 1
1007: 1
1008: 1
1009: 2
1010: 0
1011: 1
1012: 5
1013: 5
1014: 5
1015: 5
1016: 0
1017: 1
1018: 0
1019: 1
1020: 5
1021: 5
1022: 0
1023: 0
1024: 1
1025: 1
1026: 1
1027: 0
1028: 1
1029: 1
1030: 1
1031: 1
1032: 0
1033: 5
1034: 3
"""

# Parse grades using regular expression
grade_pattern = re.compile(r'(?:ID:?\s*)?(\d+)\s*[\s,:-]+\s*([0-5])', re.IGNORECASE)
parsed_grades = {}

for line in ai_output_raw.strip().split('\n'):
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    match = grade_pattern.search(line)
    if match:
        item_id = int(match.group(1))
        score = int(match.group(2))
        parsed_grades[item_id] = score

print(f"Parsed {len(parsed_grades)} grades from the AI output.")

if len(parsed_grades) > 0:
    pool_df['relevance'] = pool_df['id'].map(parsed_grades)
    missing_count = pool_df['relevance'].isna().sum()
    if missing_count > 0:
        print(f"Warning: {missing_count} rows were not graded. Setting their relevance to 0.")
        pool_df['relevance'] = pool_df['relevance'].fillna(0)
    else:
        print("All rows successfully graded!")
        
    pool_df['relevance'] = pool_df['relevance'].astype(int)
    output_df = pool_df[['query', 'movie_title', 'relevance']].copy()
    
    output_csv = 'qrels_to_grade.csv'
    output_df.to_csv(output_csv, index=False)
    print(f"Successfully exported final graded qrels to '{output_csv}' with {len(output_df)} rows.")
else:
    print("No grades were parsed. Please paste the AI output in 'ai_output_raw' or write to 'ai_grades.txt' and rerun this cell.")

Parsed 1034 grades from the AI output.
Successfully exported final graded qrels to 'qrels_to_grade.csv' with 1033 rows.
